In [ ]:
"""
Support Vector Regression with a LINEAR kernel.
"""

import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load data
df = pd.read_csv("//content/Dengue_Climate_Bangladesh.csv")
df.columns = [c.strip().upper() for c in df.columns]
df = df.sort_values(["YEAR", "MONTH"]).reset_index(drop=True)

# 2. Feature engineering
df["MONTH_SIN"] = np.sin(2 * np.pi * df["MONTH"] / 12)
df["MONTH_COS"] = np.cos(2 * np.pi * df["MONTH"] / 12)
df["CASES_LAG1"]  = df["DENGUE"].shift(1)
df["CASES_LAG2"]  = df["DENGUE"].shift(2)
df["CASES_LAG3"]  = df["DENGUE"].shift(3)
df["CASES_LAG12"] = df["DENGUE"].shift(12)
df["CASES_ROLL3"] = df["DENGUE"].shift(1).rolling(window=3).mean()

for c in ["CASES_LAG1", "CASES_LAG2", "CASES_LAG3", "CASES_LAG12", "CASES_ROLL3"]:
    df[c] = np.log1p(df[c])

df["DENGUE_LOG"] = np.log1p(df["DENGUE"])
df = df.dropna().reset_index(drop=True)

feature_cols = [
    "MIN", "MAX", "HUMIDITY", "RAINFALL",
    "MONTH_SIN", "MONTH_COS",
    "CASES_LAG1", "CASES_LAG2", "CASES_LAG3", "CASES_LAG12", "CASES_ROLL3",
]
target_col = "DENGUE_LOG"

train_df = df[df["YEAR"] < 2025].copy()
test_df  = df[(df["YEAR"] == 2025) & (df["MONTH"] <= 10)].copy()

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test,  y_test  = test_df[feature_cols],  test_df[target_col]

# 3. Pipeline + hyperparameter tuning (C, epsilon)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="linear")),
])

param_grid = {
    "svr__C":       [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 100],
    "svr__epsilon": [0.01, 0.05, 0.1, 0.2, 0.3],
}

tscv = TimeSeriesSplit(n_splits=5)
grid = GridSearchCV(
    pipe, param_grid, cv=tscv,
    scoring="neg_mean_absolute_error", n_jobs=-1
)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("Best params:", grid.best_params_)

# 4. Inspect coefficients (linear kernel -> directly interpretable)
coefs = best_model.named_steps["svr"].coef_[0]
coef_table = (
    pd.DataFrame({"feature": feature_cols, "coef": coefs})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
)
print("\n=== SVR (linear kernel) coefficients, sorted by importance ===")
print(coef_table.drop(columns="abs_coef").to_string(index=False))

# 5. Predict on 2025 Jan-Oct, back-transform from log1p
pred_log = best_model.predict(X_test)
pred_cases = np.clip(np.expm1(pred_log), 0, None)
actual_cases = np.expm1(y_test.values)

results = test_df[["YEAR", "MONTH"]].copy()
results["ACTUAL_CASES"] = actual_cases.round(0).astype(int)
results["PREDICTED_CASES"] = pred_cases.round(0).astype(int)
results["ABS_ERROR"] = (results["ACTUAL_CASES"] - results["PREDICTED_CASES"]).abs()
results["PCT_ERROR_%"] = (results["ABS_ERROR"] / results["ACTUAL_CASES"].replace(0, np.nan) * 100).round(1)

print("\n=== 2025 Jan-Oct: Actual vs Predicted ===")
print(results.to_string(index=False))

# 6. Accuracy metrics (original case-count scale)
mae  = mean_absolute_error(actual_cases, pred_cases)
rmse = np.sqrt(mean_squared_error(actual_cases, pred_cases))
r2   = r2_score(actual_cases, pred_cases)
mape = np.mean(np.abs((actual_cases - pred_cases) / actual_cases)) * 100

print("\n=== Accuracy Metrics (original case-count scale) ===")
print(f"MAE   : {mae:,.1f} cases")
print(f"RMSE  : {rmse:,.1f} cases")
print(f"R^2   : {r2:.4f}")
print(f"MAPE  : {mape:.2f}%")
print(f"Accuracy (100 - MAPE): {100 - mape:.2f}%")

results.to_csv("/content/svr_2025_predictions.csv", index=False)

Best params: {'svr__C': 1, 'svr__epsilon': 0.1}

=== SVR (linear kernel) coefficients, sorted by importance ===
    feature      coef
 CASES_LAG1  2.676280
 CASES_LAG2 -0.816704
CASES_ROLL3  0.586878
  MONTH_COS -0.390167
  MONTH_SIN -0.294950
CASES_LAG12  0.285945
   RAINFALL  0.251050
   HUMIDITY -0.218053
        MIN  0.154615
 CASES_LAG3  0.064360
        MAX -0.030586

=== 2025 Jan-Oct: Actual vs Predicted ===
 YEAR  MONTH  ACTUAL_CASES  PREDICTED_CASES  ABS_ERROR  PCT_ERROR_%
 2025      1          1161             2689       1528        131.6
 2025      2           374              533        159         42.5
 2025      3           336              396         60         17.9
 2025      4           701              387        314         44.8
 2025      5          1773             1036        737         41.6
 2025      6          5951             3159       2792         46.9
 2025      7         10684            12732       2048         19.2
 2025      8         10496           